In [1]:
import sys
sys.path.append("/home/annierosen16")  # add directory with func.py
import pandas as pd
from func import cal_buoyancy
from sklearn.metrics import mean_squared_error
import glob
import os
import re
import traceback
from scipy.interpolate import interp1d
import requests
import datetime
import metpy.calc as mpcalc
from metpy.units import units
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, SkewT
from scipy.interpolate import interp1d
from scipy.optimize import bisect
import numpy as np
import pickle
import numpy as np
import math
import pint
from pydoc import help
import xarray as xr
import netCDF4 as nc
import re
import matplotlib.pyplot as plt
from numpy import load
from numpy import asarray
from numpy import save
import pytz
from scipy import stats
from sklearn.linear_model import LinearRegression
from collections import Counter
import pymannkendall as mk
%matplotlib inline

In [2]:
# # reading in ERA5 specific humidity  

# q_path = '/home/annierosen16/data/era5sgp/era5_ml_q_full_sgp'

# file_pattern = f"{q_path}/full_q_profile_*"

# file_list = glob.glob(file_pattern)

# all_q = xr.open_mfdataset(file_list, combine='by_coords', chunks={'time': 24})

# all_q = all_q.assign_coords(valid_time=all_q.valid_time - pd.Timedelta(hours=6))

# qdf = all_q.sel(valid_time=all_q.valid_time[all_q.valid_time.dt.hour == 12])

# qdf['valid_time'].shape, qdf['pressure_level'].data.shape

In [2]:
# changing q path testing 

# first q path, yizhous data -> 05-2001 -> 08-2022

# oriented bottom to top 

qpath1 = "/data/rong4/Data/ERA5/3hourly/quvw_US/specific_humidity"

qfiles1 = sorted(glob.glob(f"{qpath1}/era5.specific_humidity.*.nc"))

q1 = xr.open_mfdataset(qfiles1, combine="by_coords", chunks={"time": 24})

# --- shift to local time (UTC → LST) ---

q1 = q1.assign_coords(time=q1.time - pd.Timedelta(hours=6))

# --- time filter: 1979–2022, months 5–9 ---

q1 = q1.sel(time=slice("1979-05-01", "2022-08-31"))

q1 = q1.sel(time=q1.time.dt.month.isin([5, 6, 7, 8, 9]))

q1 = q1.sortby("level", ascending=False)

# --- keep only 12 LST ---

q1 = q1.sel(time=q1.time.dt.hour == 12)

# --- spatial subset ---

q1 = q1.sel(latitude=slice(39, 30), longitude=slice(-105, -95))

qpath2 = os.path.expanduser('~/data/era5sgp/era5_ml_q_full_sgp_2022_2025')

qfiles2 = sorted(glob.glob(f"{qpath2}/*.nc"))

q2 = xr.open_mfdataset(qfiles2, combine="by_coords", chunks={"time": 31})

# --- standardize time coord name if needed ---

if "valid_time" in q2.coords and "time" not in q2.coords:
    
    q2 = q2.rename({"valid_time": "time"})
    
# --- standardize pressure level name ---

if "pressure_level" in q2.coords and "level" not in q2.coords:
    
    q2 = q2.rename({"pressure_level": "level"})

# --- shift UTC -> LST ---

q2 = q2.assign_coords(time=q2.time - pd.Timedelta(hours=6))

# --- keep only 12 LST ---

q2 = q2.sel(time=q2.time.dt.hour == 12)

# --- same spatial subset ---

q2 = q2.sel(latitude=slice(39, 30), longitude=slice(-105, -95))

q2 = q2.sortby("level", ascending=False)

# make expver consistent

q1 = q1.drop_vars("expver", errors="ignore")

q2 = q2.drop_vars("expver", errors="ignore")

qdf = xr.concat([q1, q2], dim="time").sortby("time")

qdf['time'].shape, qdf['level'].shape

((7191,), (29,))

In [6]:
# reading in ERA5 geopotential height & temperature

# tz_path = '/home/annierosen16/data/era5sgp/era5_ml_t_geopotential_full_sgp'

tz_path = '/home/annierosen16/data/era5sgp/era5_ml_temperature_and_geopotential_full_sgp'

file_pattern = f"{tz_path}/*temperature_and_geopotential*.nc"

file_list = glob.glob(file_pattern)

all_tz = xr.open_mfdataset(file_list, combine='by_coords', chunks={'time': 24})

all_tz = all_tz.assign_coords(valid_time=all_tz.valid_time - pd.Timedelta(hours=6))

all_tz = all_tz.sel(pressure_level=all_tz.pressure_level >= 50)

# keep only May–September AND 12 LST

tzdf = all_tz.sel(valid_time=all_tz.valid_time[
        
    (all_tz.valid_time.dt.hour == 12) &
    
    (all_tz.valid_time.dt.month.isin([5, 6, 7, 8, 9]))])

# rename pressure coordinate

tzdf = tzdf.rename({"pressure_level": "level"})

# ensure level is descending (e.g., 1000 → 50 hPa)

tzdf = tzdf.sortby("level", ascending=False)

tzdf['valid_time'].shape, tzdf['level'].data.shape

((7191,), (29,))

In [5]:
# # ERA5 information: surface pressure

# # you will have to do the same treatment for q file loading here  

# sfc_pressure_path = '/data/rong4/Data/ERA5/3hourly/quvw_US/surface_pressure'

# all_files = glob.glob(os.path.join(sfc_pressure_path, "era5.surface_pressure.*.nc"))

# # Filter files for years 2001-2018 and months 5-9

# filtered_files = [
    
#     file for file in all_files 
    
#     if 2001 <= int(os.path.basename(file).split(".")[2][:4]) <= 2018  
    
#     and 5 <= int(os.path.basename(file).split(".")[2][4:]) <= 9       
# ]

# all_sfc_p = xr.open_mfdataset(filtered_files, combine='by_coords')

# all_sfc_p = all_sfc_p.sel(latitude=slice(39.0, 30.0), longitude=slice(-105.0, -95.0))

# sfcp = all_sfc_p.assign_coords(time=all_sfc_p.time - pd.Timedelta(hours=6))

# sfcp = sfcp.sel(time=sfcp.time[sfcp.time.dt.hour == 12])

# sfcp['time'].shape

In [9]:
sfc_path1 = "/data/rong4/Data/ERA5/3hourly/quvw_US/surface_pressure"

files1 = sorted(glob.glob(os.path.join(sfc_path1, "era5.surface_pressure.*.nc")))

filtered1 = [
    
    f for f in files1
    
    if 1979 <= int(os.path.basename(f).split(".")[2][:4]) <= 2022
    
    and 5 <= int(os.path.basename(f).split(".")[2][4:]) <= 9
]

sp1 = xr.open_mfdataset(filtered1, combine="by_coords", chunks={"time": 24})

# UTC -> LST

sp1 = sp1.assign_coords(time=sp1.time - pd.Timedelta(hours=6))

# keep only 12 LST

sp1 = sp1.sel(time=sp1.time.dt.hour == 12)

# spatial subset

sp1 = sp1.sel(latitude=slice(39, 30), longitude=slice(-105, -95))

# drop expver if present

sp1 = sp1.drop_vars("expver", errors="ignore")

sfc_path2 = "/home/annierosen16/data/era5sgp/era5_sl_surface_terms_full_sgp"

files2 = sorted(glob.glob(os.path.join(sfc_path2, "*.nc")))

sp2 = xr.open_mfdataset(files2, combine="by_coords", chunks={"time": 31})

sp2 = sp2[["sp"]]

# standardize time coord name if needed

if "valid_time" in sp2.coords and "time" not in sp2.coords:
    
    sp2 = sp2.rename({"valid_time": "time"})

sp2 = sp2.assign_coords(time=sp2.time - pd.Timedelta(hours=6))

# select Sep 2022 through Sep 2025 first (as you requested)

sp2 = sp2.sel(time=slice("2022-09-01", "2025-09-30"))

# then warm-season filter (May–Sep)

sp2 = sp2.sel(time=sp2.time.dt.month.isin([5, 6, 7, 8, 9]))

# keep only 12 LST

sp2 = sp2.sel(time=sp2.time.dt.hour == 12)

# spatial subset

sp2 = sp2.sel(latitude=slice(39, 30), longitude=slice(-105, -95))

# drop expver if present

sp2 = sp2.drop_vars("expver", errors="ignore")

sfcp = xr.concat([sp1, sp2], dim="time").sortby("time")

sfcp['time'].shape

(7191,)

In [7]:
# # reading in ERA5 planetary boundary layer height  

# pblh_path = '/home/annierosen16/data/era5sgp/era5_sl_pblh_full_sgp'

# file_pattern = f"{pblh_path}/pblh_*"

# file_list = glob.glob(file_pattern)

# all_pblh = xr.open_mfdataset(file_list, combine='by_coords', chunks={'time': 24})

# all_pblh = all_pblh.assign_coords(valid_time=all_pblh.valid_time - pd.Timedelta(hours=6))

# pblhdf = all_pblh.sel(valid_time=all_pblh.valid_time[all_pblh.valid_time.dt.hour == 12])

# pblhdf = pblhdf.where((pblhdf.longitude >= -105) & (pblhdf.longitude <= -95), drop=True)

# pblhdf['valid_time'].shape

In [12]:
# loading all necessary information for buoyancy 

g = 9.80665 # m/s^2

# 1 dimensional

times = qdf['time'].data

# 3 dimensional (time, lat, lon)

# pblh = pblhdf['blh'].load().data

# 3 dimensional (time, lat, lon)

sp = sfcp['sp'].load().data / 100 # conversion to hPa

# 4 dimensional (time, pressure, lat, lon)

q = qdf['q'].load().data * 1000 # conversion to g/kg

# 4 dimensional (time, pressure - reversed to go from bottom to top -, lat, lon)

# t = tzdf['t'].load().data[:, ::-1, :, :] - 273.15 # conversion to Celsius

t = tzdf['t'].load().data - 273.15 # conversion to Celsius

# 4 dimensional (time, pressure - reversed to go from bottom to top -, lat, lon)

# z = tzdf['z'].load().data[:, ::-1, :, :] / g # geopotential height

z = tzdf['z'].load().data / g # geopotential height

lats = tzdf['latitude'].data

lons = tzdf['longitude'].data

In [38]:
# step 1 -> interpolate pressure to altitude

def interpolate_pressure_to_altitude(pres_array, alt_array, surface_pressure):
    
    x = alt_array
    
    y = pres_array

    f = interp1d(x, y, assume_sorted=False, fill_value="extrapolate")

    max_x = np.nanmax(x)
    
    min_x = np.nanmin(x)

    xint = np.arange(min_x, max_x, 20)

    # Interpolate pressure against new altitude intervals
    
    yint = f(xint) * units.hPa
    
    yint = yint.magnitude  # Extract numerical values

    # Compute altitude above surface
    
    mask = yint <= surface_pressure
    
    xnew = xint[mask]
    
    ynew = yint[mask]

    if xnew.size == 0:
        
        return np.array([]), np.array([]), np.nan 

    sfc_alt = xnew[0]  # Define surface altitude before adjusting xnew
    
    xnew_rel = xnew - sfc_alt

    # Apply single mask to keep values within 5000m AGL
    
    valid_mask = xnew_rel <= 4000 # meters AGL
    
    xclean = xnew_rel[valid_mask]
    
    if xclean[-1] != 4000:
        
        print(xclean[-1])
    
    yclean = ynew[valid_mask]

    return yclean, sfc_alt

In [41]:
# Get years from datetime array

years = pd.to_datetime(times).year

# Boolean mask for 1979–2000

year_mask = (years >= 1979) & (years <= 2000)

# Get indices that satisfy condition

valid_time_indices = np.where(year_mask)[0]

In [ ]:
# for all days (at 12 LST), latitudes & longitudes: interpolate pressure 

num_levels = 201 # equivalent to 4000 meters AGL for every gridpoint

num_times = times.shape[0]

# from bottom to top 

pres_array = tzdf['level'].data

interpolated_pressure = np.full((num_times, num_levels, lats.shape[0], lons.shape[0]), np.nan)

# surface_altitudes captures geopotential height at the surface for all times and gridpoints

surface_altitudes = np.full((num_times, lats.shape[0], lons.shape[0]), np.nan)

# for tidx in range(times.shape[0]):

for tidx in valid_time_indices:
    
    print(times[tidx])

    for latidx in range(lats.shape[0]):

        for lonidx in range(lons.shape[0]):

            alt_array = z[tidx, :, latidx, lonidx]

            sfc_pres = sp[tidx, latidx, lonidx]

            new_pres, sfc_alt = interpolate_pressure_to_altitude(pres_array, alt_array, sfc_pres)

            interpolated_pressure[tidx, :, latidx, lonidx] = new_pres

            surface_altitudes[tidx, latidx, lonidx] = sfc_alt

1979-05-01T12:00:00.000000000
1979-05-02T12:00:00.000000000
1979-05-03T12:00:00.000000000
1979-05-04T12:00:00.000000000
1979-05-05T12:00:00.000000000
1979-05-06T12:00:00.000000000
1979-05-07T12:00:00.000000000
1979-05-08T12:00:00.000000000
1979-05-09T12:00:00.000000000
1979-05-10T12:00:00.000000000
1979-05-11T12:00:00.000000000
1979-05-12T12:00:00.000000000
1979-05-13T12:00:00.000000000
1979-05-14T12:00:00.000000000
1979-05-15T12:00:00.000000000
1979-05-16T12:00:00.000000000
1979-05-17T12:00:00.000000000
1979-05-18T12:00:00.000000000
1979-05-19T12:00:00.000000000
1979-05-20T12:00:00.000000000
1979-05-21T12:00:00.000000000
1979-05-22T12:00:00.000000000
1979-05-23T12:00:00.000000000
1979-05-24T12:00:00.000000000
1979-05-25T12:00:00.000000000
1979-05-26T12:00:00.000000000
1979-05-27T12:00:00.000000000
1979-05-28T12:00:00.000000000
1979-05-29T12:00:00.000000000
1979-05-30T12:00:00.000000000
1979-05-31T12:00:00.000000000
1979-06-01T12:00:00.000000000
1979-06-02T12:00:00.000000000
1979-06-03

1980-08-30T12:00:00.000000000
1980-08-31T12:00:00.000000000
1980-09-01T12:00:00.000000000
1980-09-02T12:00:00.000000000
1980-09-03T12:00:00.000000000
1980-09-04T12:00:00.000000000
1980-09-05T12:00:00.000000000
1980-09-06T12:00:00.000000000
1980-09-07T12:00:00.000000000
1980-09-08T12:00:00.000000000
1980-09-09T12:00:00.000000000
1980-09-10T12:00:00.000000000
1980-09-11T12:00:00.000000000
1980-09-12T12:00:00.000000000
1980-09-13T12:00:00.000000000
1980-09-14T12:00:00.000000000
1980-09-15T12:00:00.000000000
1980-09-16T12:00:00.000000000
1980-09-17T12:00:00.000000000
1980-09-18T12:00:00.000000000
1980-09-19T12:00:00.000000000
1980-09-20T12:00:00.000000000
1980-09-21T12:00:00.000000000
1980-09-22T12:00:00.000000000
1980-09-23T12:00:00.000000000
1980-09-24T12:00:00.000000000
1980-09-25T12:00:00.000000000
1980-09-26T12:00:00.000000000
1980-09-27T12:00:00.000000000
1980-09-28T12:00:00.000000000
1980-09-29T12:00:00.000000000
1980-09-30T12:00:00.000000000
1981-05-01T12:00:00.000000000
1981-05-02

1982-07-29T12:00:00.000000000
1982-07-30T12:00:00.000000000
1982-07-31T12:00:00.000000000
1982-08-01T12:00:00.000000000
1982-08-02T12:00:00.000000000
1982-08-03T12:00:00.000000000
1982-08-04T12:00:00.000000000
1982-08-05T12:00:00.000000000
1982-08-06T12:00:00.000000000
1982-08-07T12:00:00.000000000
1982-08-08T12:00:00.000000000
1982-08-09T12:00:00.000000000
1982-08-10T12:00:00.000000000
1982-08-11T12:00:00.000000000
1982-08-12T12:00:00.000000000
1982-08-13T12:00:00.000000000
1982-08-14T12:00:00.000000000
1982-08-15T12:00:00.000000000
1982-08-16T12:00:00.000000000
1982-08-17T12:00:00.000000000
1982-08-18T12:00:00.000000000
1982-08-19T12:00:00.000000000
1982-08-20T12:00:00.000000000
1982-08-21T12:00:00.000000000
1982-08-22T12:00:00.000000000
1982-08-23T12:00:00.000000000
1982-08-24T12:00:00.000000000
1982-08-25T12:00:00.000000000
1982-08-26T12:00:00.000000000
1982-08-27T12:00:00.000000000
1982-08-28T12:00:00.000000000
1982-08-29T12:00:00.000000000
1982-08-30T12:00:00.000000000
1982-08-31

1984-06-27T12:00:00.000000000
1984-06-28T12:00:00.000000000
1984-06-29T12:00:00.000000000
1984-06-30T12:00:00.000000000
1984-07-01T12:00:00.000000000
1984-07-02T12:00:00.000000000
1984-07-03T12:00:00.000000000
1984-07-04T12:00:00.000000000
1984-07-05T12:00:00.000000000
1984-07-06T12:00:00.000000000
1984-07-07T12:00:00.000000000
1984-07-08T12:00:00.000000000
1984-07-09T12:00:00.000000000
1984-07-10T12:00:00.000000000
1984-07-11T12:00:00.000000000
1984-07-12T12:00:00.000000000
1984-07-13T12:00:00.000000000
1984-07-14T12:00:00.000000000
1984-07-15T12:00:00.000000000
1984-07-16T12:00:00.000000000
1984-07-17T12:00:00.000000000
1984-07-18T12:00:00.000000000
1984-07-19T12:00:00.000000000
1984-07-20T12:00:00.000000000
1984-07-21T12:00:00.000000000
1984-07-22T12:00:00.000000000
1984-07-23T12:00:00.000000000
1984-07-24T12:00:00.000000000
1984-07-25T12:00:00.000000000
1984-07-26T12:00:00.000000000
1984-07-27T12:00:00.000000000
1984-07-28T12:00:00.000000000
1984-07-29T12:00:00.000000000
1984-07-30

1986-05-26T12:00:00.000000000
1986-05-27T12:00:00.000000000
1986-05-28T12:00:00.000000000
1986-05-29T12:00:00.000000000
1986-05-30T12:00:00.000000000
1986-05-31T12:00:00.000000000
1986-06-01T12:00:00.000000000
1986-06-02T12:00:00.000000000
1986-06-03T12:00:00.000000000
1986-06-04T12:00:00.000000000
1986-06-05T12:00:00.000000000
1986-06-06T12:00:00.000000000
1986-06-07T12:00:00.000000000
1986-06-08T12:00:00.000000000
1986-06-09T12:00:00.000000000
1986-06-10T12:00:00.000000000
1986-06-11T12:00:00.000000000
1986-06-12T12:00:00.000000000
1986-06-13T12:00:00.000000000
1986-06-14T12:00:00.000000000
1986-06-15T12:00:00.000000000
1986-06-16T12:00:00.000000000
1986-06-17T12:00:00.000000000
1986-06-18T12:00:00.000000000
1986-06-19T12:00:00.000000000
1986-06-20T12:00:00.000000000
1986-06-21T12:00:00.000000000
1986-06-22T12:00:00.000000000
1986-06-23T12:00:00.000000000
1986-06-24T12:00:00.000000000
1986-06-25T12:00:00.000000000
1986-06-26T12:00:00.000000000
1986-06-27T12:00:00.000000000
1986-06-28

1987-09-24T12:00:00.000000000
1987-09-25T12:00:00.000000000
1987-09-26T12:00:00.000000000
1987-09-27T12:00:00.000000000
1987-09-28T12:00:00.000000000
1987-09-29T12:00:00.000000000
1987-09-30T12:00:00.000000000
1988-05-01T12:00:00.000000000
1988-05-02T12:00:00.000000000
1988-05-03T12:00:00.000000000
1988-05-04T12:00:00.000000000
1988-05-05T12:00:00.000000000
1988-05-06T12:00:00.000000000
1988-05-07T12:00:00.000000000
1988-05-08T12:00:00.000000000
1988-05-09T12:00:00.000000000
1988-05-10T12:00:00.000000000
1988-05-11T12:00:00.000000000
1988-05-12T12:00:00.000000000
1988-05-13T12:00:00.000000000
1988-05-14T12:00:00.000000000
1988-05-15T12:00:00.000000000
1988-05-16T12:00:00.000000000
1988-05-17T12:00:00.000000000
1988-05-18T12:00:00.000000000
1988-05-19T12:00:00.000000000
1988-05-20T12:00:00.000000000
1988-05-21T12:00:00.000000000
1988-05-22T12:00:00.000000000
1988-05-23T12:00:00.000000000
1988-05-24T12:00:00.000000000
1988-05-25T12:00:00.000000000
1988-05-26T12:00:00.000000000
1988-05-27

1989-08-23T12:00:00.000000000
1989-08-24T12:00:00.000000000
1989-08-25T12:00:00.000000000
1989-08-26T12:00:00.000000000
1989-08-27T12:00:00.000000000
1989-08-28T12:00:00.000000000
1989-08-29T12:00:00.000000000
1989-08-30T12:00:00.000000000
1989-08-31T12:00:00.000000000
1989-09-01T12:00:00.000000000


In [51]:
# Subset interpolated pressure
interpolated_pressure = interpolated_pressure[valid_time_indices, :, :, :]

# Subset surface altitudes
surface_altitudes = surface_altitudes[valid_time_indices, :, :]

In [54]:
# cleaner version

def interpolate_var_to_altitude(variable_to_interpolate, alt_array, sfc_alt):
    
    x = alt_array
    
    y = variable_to_interpolate

    f = interp1d(x, y, assume_sorted=False, fill_value="extrapolate")

    max_x = np.nanmax(x)
    
    min_x = np.nanmin(x)

    xnew = np.arange(min_x, max_x, 20)
    
    ynew = f(xnew)

    # Compute relative altitude
    
    xnew_rel = xnew - sfc_alt
    
    valid_mask = np.logical_and(xnew_rel >= 0, xnew_rel <= 4000)

    yfinal = ynew[valid_mask]
    
    xfinal = xnew_rel[valid_mask]

    return yfinal

In [55]:
# interpolation of specific humidity 

num_levels = 201 # equivalent to 4000 meters AGL for every gridpoint

num_times = times.shape[0]

interpolated_q = np.full((num_times, num_levels, lats.shape[0], lons.shape[0]), np.nan)

# for tidx in range(times.shape[0]):
    
for tidx in valid_time_indices:
    
    print(times[tidx])

    for latidx in range(lats.shape[0]):

        for lonidx in range(lons.shape[0]):
            
            q_array = q[tidx, :, latidx, lonidx]

            alt_array = z[tidx, :, latidx, lonidx]

            sfc_alt = surface_altitudes[tidx, latidx, lonidx]

            new_q = interpolate_var_to_altitude(q_array, alt_array, sfc_alt)

            interpolated_q[tidx, :, latidx, lonidx] = new_q

1979-05-01T12:00:00.000000000
1979-05-02T12:00:00.000000000
1979-05-03T12:00:00.000000000
1979-05-04T12:00:00.000000000
1979-05-05T12:00:00.000000000
1979-05-06T12:00:00.000000000
1979-05-07T12:00:00.000000000
1979-05-08T12:00:00.000000000
1979-05-09T12:00:00.000000000
1979-05-10T12:00:00.000000000
1979-05-11T12:00:00.000000000
1979-05-12T12:00:00.000000000
1979-05-13T12:00:00.000000000
1979-05-14T12:00:00.000000000
1979-05-15T12:00:00.000000000
1979-05-16T12:00:00.000000000
1979-05-17T12:00:00.000000000
1979-05-18T12:00:00.000000000
1979-05-19T12:00:00.000000000
1979-05-20T12:00:00.000000000
1979-05-21T12:00:00.000000000
1979-05-22T12:00:00.000000000
1979-05-23T12:00:00.000000000
1979-05-24T12:00:00.000000000
1979-05-25T12:00:00.000000000
1979-05-26T12:00:00.000000000
1979-05-27T12:00:00.000000000
1979-05-28T12:00:00.000000000
1979-05-29T12:00:00.000000000
1979-05-30T12:00:00.000000000
1979-05-31T12:00:00.000000000
1979-06-01T12:00:00.000000000
1979-06-02T12:00:00.000000000
1979-06-03

1980-08-30T12:00:00.000000000
1980-08-31T12:00:00.000000000
1980-09-01T12:00:00.000000000
1980-09-02T12:00:00.000000000
1980-09-03T12:00:00.000000000
1980-09-04T12:00:00.000000000
1980-09-05T12:00:00.000000000
1980-09-06T12:00:00.000000000
1980-09-07T12:00:00.000000000
1980-09-08T12:00:00.000000000
1980-09-09T12:00:00.000000000
1980-09-10T12:00:00.000000000
1980-09-11T12:00:00.000000000
1980-09-12T12:00:00.000000000
1980-09-13T12:00:00.000000000
1980-09-14T12:00:00.000000000
1980-09-15T12:00:00.000000000
1980-09-16T12:00:00.000000000
1980-09-17T12:00:00.000000000
1980-09-18T12:00:00.000000000
1980-09-19T12:00:00.000000000
1980-09-20T12:00:00.000000000
1980-09-21T12:00:00.000000000
1980-09-22T12:00:00.000000000
1980-09-23T12:00:00.000000000
1980-09-24T12:00:00.000000000
1980-09-25T12:00:00.000000000
1980-09-26T12:00:00.000000000
1980-09-27T12:00:00.000000000
1980-09-28T12:00:00.000000000
1980-09-29T12:00:00.000000000
1980-09-30T12:00:00.000000000
1981-05-01T12:00:00.000000000
1981-05-02

1982-07-29T12:00:00.000000000
1982-07-30T12:00:00.000000000
1982-07-31T12:00:00.000000000
1982-08-01T12:00:00.000000000
1982-08-02T12:00:00.000000000
1982-08-03T12:00:00.000000000
1982-08-04T12:00:00.000000000
1982-08-05T12:00:00.000000000
1982-08-06T12:00:00.000000000
1982-08-07T12:00:00.000000000
1982-08-08T12:00:00.000000000
1982-08-09T12:00:00.000000000
1982-08-10T12:00:00.000000000
1982-08-11T12:00:00.000000000
1982-08-12T12:00:00.000000000
1982-08-13T12:00:00.000000000
1982-08-14T12:00:00.000000000
1982-08-15T12:00:00.000000000
1982-08-16T12:00:00.000000000
1982-08-17T12:00:00.000000000
1982-08-18T12:00:00.000000000
1982-08-19T12:00:00.000000000
1982-08-20T12:00:00.000000000
1982-08-21T12:00:00.000000000
1982-08-22T12:00:00.000000000
1982-08-23T12:00:00.000000000
1982-08-24T12:00:00.000000000
1982-08-25T12:00:00.000000000
1982-08-26T12:00:00.000000000
1982-08-27T12:00:00.000000000
1982-08-28T12:00:00.000000000
1982-08-29T12:00:00.000000000
1982-08-30T12:00:00.000000000
1982-08-31

1984-06-27T12:00:00.000000000
1984-06-28T12:00:00.000000000
1984-06-29T12:00:00.000000000
1984-06-30T12:00:00.000000000
1984-07-01T12:00:00.000000000
1984-07-02T12:00:00.000000000
1984-07-03T12:00:00.000000000
1984-07-04T12:00:00.000000000
1984-07-05T12:00:00.000000000
1984-07-06T12:00:00.000000000
1984-07-07T12:00:00.000000000
1984-07-08T12:00:00.000000000
1984-07-09T12:00:00.000000000
1984-07-10T12:00:00.000000000
1984-07-11T12:00:00.000000000
1984-07-12T12:00:00.000000000
1984-07-13T12:00:00.000000000
1984-07-14T12:00:00.000000000
1984-07-15T12:00:00.000000000
1984-07-16T12:00:00.000000000
1984-07-17T12:00:00.000000000
1984-07-18T12:00:00.000000000
1984-07-19T12:00:00.000000000
1984-07-20T12:00:00.000000000
1984-07-21T12:00:00.000000000
1984-07-22T12:00:00.000000000
1984-07-23T12:00:00.000000000
1984-07-24T12:00:00.000000000
1984-07-25T12:00:00.000000000
1984-07-26T12:00:00.000000000
1984-07-27T12:00:00.000000000
1984-07-28T12:00:00.000000000
1984-07-29T12:00:00.000000000
1984-07-30

1986-05-26T12:00:00.000000000
1986-05-27T12:00:00.000000000
1986-05-28T12:00:00.000000000
1986-05-29T12:00:00.000000000
1986-05-30T12:00:00.000000000
1986-05-31T12:00:00.000000000
1986-06-01T12:00:00.000000000
1986-06-02T12:00:00.000000000
1986-06-03T12:00:00.000000000
1986-06-04T12:00:00.000000000
1986-06-05T12:00:00.000000000
1986-06-06T12:00:00.000000000
1986-06-07T12:00:00.000000000
1986-06-08T12:00:00.000000000
1986-06-09T12:00:00.000000000
1986-06-10T12:00:00.000000000
1986-06-11T12:00:00.000000000
1986-06-12T12:00:00.000000000
1986-06-13T12:00:00.000000000
1986-06-14T12:00:00.000000000
1986-06-15T12:00:00.000000000
1986-06-16T12:00:00.000000000
1986-06-17T12:00:00.000000000
1986-06-18T12:00:00.000000000
1986-06-19T12:00:00.000000000
1986-06-20T12:00:00.000000000
1986-06-21T12:00:00.000000000
1986-06-22T12:00:00.000000000
1986-06-23T12:00:00.000000000
1986-06-24T12:00:00.000000000
1986-06-25T12:00:00.000000000
1986-06-26T12:00:00.000000000
1986-06-27T12:00:00.000000000
1986-06-28

...

1993-06-20T12:00:00.000000000
1993-06-21T12:00:00.000000000
1993-06-22T12:00:00.000000000
1993-06-23T12:00:00.000000000
1993-06-24T12:00:00.000000000
1993-06-25T12:00:00.000000000
1993-06-26T12:00:00.000000000
1993-06-27T12:00:00.000000000
1993-06-28T12:00:00.000000000
1993-06-29T12:00:00.000000000
1993-06-30T12:00:00.000000000
1993-07-01T12:00:00.000000000
1993-07-02T12:00:00.000000000
1993-07-03T12:00:00.000000000
1993-07-04T12:00:00.000000000
1993-07-05T12:00:00.000000000
1993-07-06T12:00:00.000000000
1993-07-07T12:00:00.000000000
1993-07-08T12:00:00.000000000
1993-07-09T12:00:00.000000000
1993-07-10T12:00:00.000000000
1993-07-11T12:00:00.000000000
1993-07-12T12:00:00.000000000
1993-07-13T12:00:00.000000000
1993-07-14T12:00:00.000000000
1993-07-15T12:00:00.000000000
1993-07-16T12:00:00.000000000
1993-07-17T12:00:00.000000000
1993-07-18T12:00:00.000000000
1993-07-19T12:00:00.000000000
1993-07-20T12:00:00.000000000
1993-07-21T12:00:00.000000000
1993-07-22T12:00:00.000000000
1993-07-23

1995-05-19T12:00:00.000000000
1995-05-20T12:00:00.000000000
1995-05-21T12:00:00.000000000
1995-05-22T12:00:00.000000000
1995-05-23T12:00:00.000000000
1995-05-24T12:00:00.000000000
1995-05-25T12:00:00.000000000
1995-05-26T12:00:00.000000000
1995-05-27T12:00:00.000000000
1995-05-28T12:00:00.000000000
1995-05-29T12:00:00.000000000
1995-05-30T12:00:00.000000000
1995-05-31T12:00:00.000000000
1995-06-01T12:00:00.000000000
1995-06-02T12:00:00.000000000
1995-06-03T12:00:00.000000000
1995-06-04T12:00:00.000000000
1995-06-05T12:00:00.000000000
1995-06-06T12:00:00.000000000
1995-06-07T12:00:00.000000000
1995-06-08T12:00:00.000000000
1995-06-09T12:00:00.000000000
1995-06-10T12:00:00.000000000
1995-06-11T12:00:00.000000000
1995-06-12T12:00:00.000000000
1995-06-13T12:00:00.000000000
1995-06-14T12:00:00.000000000
1995-06-15T12:00:00.000000000
1995-06-16T12:00:00.000000000
1995-06-17T12:00:00.000000000
1995-06-18T12:00:00.000000000
1995-06-19T12:00:00.000000000
1995-06-20T12:00:00.000000000
1995-06-21

1996-09-17T12:00:00.000000000
1996-09-18T12:00:00.000000000
1996-09-19T12:00:00.000000000
1996-09-20T12:00:00.000000000
1996-09-21T12:00:00.000000000
1996-09-22T12:00:00.000000000
1996-09-23T12:00:00.000000000
1996-09-24T12:00:00.000000000
1996-09-25T12:00:00.000000000
1996-09-26T12:00:00.000000000
1996-09-27T12:00:00.000000000
1996-09-28T12:00:00.000000000
1996-09-29T12:00:00.000000000
1996-09-30T12:00:00.000000000
1997-05-01T12:00:00.000000000
1997-05-02T12:00:00.000000000
1997-05-03T12:00:00.000000000
1997-05-04T12:00:00.000000000
1997-05-05T12:00:00.000000000
1997-05-06T12:00:00.000000000
1997-05-07T12:00:00.000000000
1997-05-08T12:00:00.000000000
1997-05-09T12:00:00.000000000
1997-05-10T12:00:00.000000000
1997-05-11T12:00:00.000000000
1997-05-12T12:00:00.000000000
1997-05-13T12:00:00.000000000
1997-05-14T12:00:00.000000000
1997-05-15T12:00:00.000000000
1997-05-16T12:00:00.000000000
1997-05-17T12:00:00.000000000
1997-05-18T12:00:00.000000000
1997-05-19T12:00:00.000000000
1997-05-20

1998-08-16T12:00:00.000000000
1998-08-17T12:00:00.000000000
1998-08-18T12:00:00.000000000
1998-08-19T12:00:00.000000000
1998-08-20T12:00:00.000000000
1998-08-21T12:00:00.000000000
1998-08-22T12:00:00.000000000
1998-08-23T12:00:00.000000000
1998-08-24T12:00:00.000000000
1998-08-25T12:00:00.000000000
1998-08-26T12:00:00.000000000
1998-08-27T12:00:00.000000000
1998-08-28T12:00:00.000000000
1998-08-29T12:00:00.000000000
1998-08-30T12:00:00.000000000
1998-08-31T12:00:00.000000000
1998-09-01T12:00:00.000000000
1998-09-02T12:00:00.000000000
1998-09-03T12:00:00.000000000
1998-09-04T12:00:00.000000000
1998-09-05T12:00:00.000000000
1998-09-06T12:00:00.000000000
1998-09-07T12:00:00.000000000
1998-09-08T12:00:00.000000000
1998-09-09T12:00:00.000000000
1998-09-10T12:00:00.000000000
1998-09-11T12:00:00.000000000
1998-09-12T12:00:00.000000000
1998-09-13T12:00:00.000000000
1998-09-14T12:00:00.000000000
1998-09-15T12:00:00.000000000
1998-09-16T12:00:00.000000000
1998-09-17T12:00:00.000000000
1998-09-18

2000-07-15T12:00:00.000000000
2000-07-16T12:00:00.000000000
2000-07-17T12:00:00.000000000
2000-07-18T12:00:00.000000000
2000-07-19T12:00:00.000000000
2000-07-20T12:00:00.000000000
2000-07-21T12:00:00.000000000
2000-07-22T12:00:00.000000000
2000-07-23T12:00:00.000000000
2000-07-24T12:00:00.000000000
2000-07-25T12:00:00.000000000
2000-07-26T12:00:00.000000000
2000-07-27T12:00:00.000000000
2000-07-28T12:00:00.000000000
2000-07-29T12:00:00.000000000
2000-07-30T12:00:00.000000000
2000-07-31T12:00:00.000000000
2000-08-01T12:00:00.000000000
2000-08-02T12:00:00.000000000
2000-08-03T12:00:00.000000000
2000-08-04T12:00:00.000000000
2000-08-05T12:00:00.000000000
2000-08-06T12:00:00.000000000
2000-08-07T12:00:00.000000000
2000-08-08T12:00:00.000000000
2000-08-09T12:00:00.000000000
2000-08-10T12:00:00.000000000
2000-08-11T12:00:00.000000000
2000-08-12T12:00:00.000000000
2000-08-13T12:00:00.000000000
2000-08-14T12:00:00.000000000
2000-08-15T12:00:00.000000000
2000-08-16T12:00:00.000000000
2000-08-17

In [57]:
interpolated_q = interpolated_q[valid_time_indices, :, :, :]

In [59]:
# interpolation of temperature 

num_levels = 201 # equivalent to 4000 meters AGL for every gridpoint

num_times = times.shape[0]

interpolated_t = np.full((num_times, num_levels, lats.shape[0], lons.shape[0]), np.nan)

# for tidx in range(times.shape[0]):

for tidx in valid_time_indices:
    
    print(times[tidx])

    for latidx in range(lats.shape[0]):

        for lonidx in range(lons.shape[0]):
            
            t_array = t[tidx, :, latidx, lonidx]

            alt_array = z[tidx, :, latidx, lonidx]

            sfc_alt = surface_altitudes[tidx, latidx, lonidx]

            new_t = interpolate_var_to_altitude(t_array, alt_array, sfc_alt)

            interpolated_t[tidx, :, latidx, lonidx] = new_t

1979-05-01T12:00:00.000000000
1979-05-02T12:00:00.000000000
1979-05-03T12:00:00.000000000
1979-05-04T12:00:00.000000000
1979-05-05T12:00:00.000000000
1979-05-06T12:00:00.000000000
1979-05-07T12:00:00.000000000
1979-05-08T12:00:00.000000000
1979-05-09T12:00:00.000000000
1979-05-10T12:00:00.000000000
1979-05-11T12:00:00.000000000
1979-05-12T12:00:00.000000000
1979-05-13T12:00:00.000000000
1979-05-14T12:00:00.000000000
1979-05-15T12:00:00.000000000
1979-05-16T12:00:00.000000000
1979-05-17T12:00:00.000000000
1979-05-18T12:00:00.000000000
1979-05-19T12:00:00.000000000
1979-05-20T12:00:00.000000000
1979-05-21T12:00:00.000000000
1979-05-22T12:00:00.000000000
1979-05-23T12:00:00.000000000
1979-05-24T12:00:00.000000000
1979-05-25T12:00:00.000000000
1979-05-26T12:00:00.000000000
1979-05-27T12:00:00.000000000
1979-05-28T12:00:00.000000000
1979-05-29T12:00:00.000000000
1979-05-30T12:00:00.000000000
1979-05-31T12:00:00.000000000
1979-06-01T12:00:00.000000000
1979-06-02T12:00:00.000000000
1979-06-03

1980-08-30T12:00:00.000000000
1980-08-31T12:00:00.000000000
1980-09-01T12:00:00.000000000
1980-09-02T12:00:00.000000000
1980-09-03T12:00:00.000000000
1980-09-04T12:00:00.000000000
1980-09-05T12:00:00.000000000
1980-09-06T12:00:00.000000000
1980-09-07T12:00:00.000000000
1980-09-08T12:00:00.000000000
1980-09-09T12:00:00.000000000
1980-09-10T12:00:00.000000000
1980-09-11T12:00:00.000000000
1980-09-12T12:00:00.000000000
1980-09-13T12:00:00.000000000
1980-09-14T12:00:00.000000000
1980-09-15T12:00:00.000000000
1980-09-16T12:00:00.000000000
1980-09-17T12:00:00.000000000
1980-09-18T12:00:00.000000000
1980-09-19T12:00:00.000000000
1980-09-20T12:00:00.000000000
1980-09-21T12:00:00.000000000
1980-09-22T12:00:00.000000000
1980-09-23T12:00:00.000000000
1980-09-24T12:00:00.000000000
1980-09-25T12:00:00.000000000
1980-09-26T12:00:00.000000000
1980-09-27T12:00:00.000000000
1980-09-28T12:00:00.000000000
1980-09-29T12:00:00.000000000
1980-09-30T12:00:00.000000000
1981-05-01T12:00:00.000000000
1981-05-02

1982-07-29T12:00:00.000000000
1982-07-30T12:00:00.000000000
1982-07-31T12:00:00.000000000
1982-08-01T12:00:00.000000000
1982-08-02T12:00:00.000000000
1982-08-03T12:00:00.000000000
1982-08-04T12:00:00.000000000
1982-08-05T12:00:00.000000000
1982-08-06T12:00:00.000000000
1982-08-07T12:00:00.000000000
1982-08-08T12:00:00.000000000
1982-08-09T12:00:00.000000000
1982-08-10T12:00:00.000000000
1982-08-11T12:00:00.000000000
1982-08-12T12:00:00.000000000
1982-08-13T12:00:00.000000000
1982-08-14T12:00:00.000000000
1982-08-15T12:00:00.000000000
1982-08-16T12:00:00.000000000
1982-08-17T12:00:00.000000000
1982-08-18T12:00:00.000000000
1982-08-19T12:00:00.000000000
1982-08-20T12:00:00.000000000
1982-08-21T12:00:00.000000000
1982-08-22T12:00:00.000000000
1982-08-23T12:00:00.000000000
1982-08-24T12:00:00.000000000
1982-08-25T12:00:00.000000000
1982-08-26T12:00:00.000000000
1982-08-27T12:00:00.000000000
1982-08-28T12:00:00.000000000
1982-08-29T12:00:00.000000000
1982-08-30T12:00:00.000000000
1982-08-31

1984-06-27T12:00:00.000000000
1984-06-28T12:00:00.000000000
1984-06-29T12:00:00.000000000
1984-06-30T12:00:00.000000000
1984-07-01T12:00:00.000000000
1984-07-02T12:00:00.000000000
1984-07-03T12:00:00.000000000
1984-07-04T12:00:00.000000000
1984-07-05T12:00:00.000000000
1984-07-06T12:00:00.000000000
1984-07-07T12:00:00.000000000
1984-07-08T12:00:00.000000000
1984-07-09T12:00:00.000000000
1984-07-10T12:00:00.000000000
1984-07-11T12:00:00.000000000
1984-07-12T12:00:00.000000000
1984-07-13T12:00:00.000000000
1984-07-14T12:00:00.000000000
1984-07-15T12:00:00.000000000
1984-07-16T12:00:00.000000000
1984-07-17T12:00:00.000000000
1984-07-18T12:00:00.000000000
1984-07-19T12:00:00.000000000
1984-07-20T12:00:00.000000000
1984-07-21T12:00:00.000000000
1984-07-22T12:00:00.000000000
1984-07-23T12:00:00.000000000
1984-07-24T12:00:00.000000000
1984-07-25T12:00:00.000000000
1984-07-26T12:00:00.000000000
1984-07-27T12:00:00.000000000
1984-07-28T12:00:00.000000000
1984-07-29T12:00:00.000000000
1984-07-30

1986-05-26T12:00:00.000000000
1986-05-27T12:00:00.000000000
1986-05-28T12:00:00.000000000
1986-05-29T12:00:00.000000000
1986-05-30T12:00:00.000000000
1986-05-31T12:00:00.000000000
1986-06-01T12:00:00.000000000
1986-06-02T12:00:00.000000000
1986-06-03T12:00:00.000000000
1986-06-04T12:00:00.000000000
1986-06-05T12:00:00.000000000
1986-06-06T12:00:00.000000000
1986-06-07T12:00:00.000000000
1986-06-08T12:00:00.000000000
1986-06-09T12:00:00.000000000
1986-06-10T12:00:00.000000000
1986-06-11T12:00:00.000000000
1986-06-12T12:00:00.000000000
1986-06-13T12:00:00.000000000
1986-06-14T12:00:00.000000000
1986-06-15T12:00:00.000000000
1986-06-16T12:00:00.000000000
1986-06-17T12:00:00.000000000
1986-06-18T12:00:00.000000000
1986-06-19T12:00:00.000000000
1986-06-20T12:00:00.000000000
1986-06-21T12:00:00.000000000
1986-06-22T12:00:00.000000000
1986-06-23T12:00:00.000000000
1986-06-24T12:00:00.000000000
1986-06-25T12:00:00.000000000
1986-06-26T12:00:00.000000000
1986-06-27T12:00:00.000000000
1986-06-28

...

1992-09-26T12:00:00.000000000
1992-09-27T12:00:00.000000000
1992-09-28T12:00:00.000000000
1992-09-29T12:00:00.000000000
1992-09-30T12:00:00.000000000
1993-05-01T12:00:00.000000000
1993-05-02T12:00:00.000000000
1993-05-03T12:00:00.000000000
1993-05-04T12:00:00.000000000
1993-05-05T12:00:00.000000000
1993-05-06T12:00:00.000000000
1993-05-07T12:00:00.000000000
1993-05-08T12:00:00.000000000
1993-05-09T12:00:00.000000000
1993-05-10T12:00:00.000000000
1993-05-11T12:00:00.000000000
1993-05-12T12:00:00.000000000
1993-05-13T12:00:00.000000000
1993-05-14T12:00:00.000000000
1993-05-15T12:00:00.000000000
1993-05-16T12:00:00.000000000
1993-05-17T12:00:00.000000000
1993-05-18T12:00:00.000000000
1993-05-19T12:00:00.000000000
1993-05-20T12:00:00.000000000
1993-05-21T12:00:00.000000000
1993-05-22T12:00:00.000000000
1993-05-23T12:00:00.000000000
1993-05-24T12:00:00.000000000
1993-05-25T12:00:00.000000000
1993-05-26T12:00:00.000000000
1993-05-27T12:00:00.000000000
1993-05-28T12:00:00.000000000
1993-05-29

1994-08-25T12:00:00.000000000
1994-08-26T12:00:00.000000000
1994-08-27T12:00:00.000000000
1994-08-28T12:00:00.000000000
1994-08-29T12:00:00.000000000
1994-08-30T12:00:00.000000000
1994-08-31T12:00:00.000000000
1994-09-01T12:00:00.000000000
1994-09-02T12:00:00.000000000
1994-09-03T12:00:00.000000000
1994-09-04T12:00:00.000000000
1994-09-05T12:00:00.000000000
1994-09-06T12:00:00.000000000
1994-09-07T12:00:00.000000000
1994-09-08T12:00:00.000000000
1994-09-09T12:00:00.000000000
1994-09-10T12:00:00.000000000
1994-09-11T12:00:00.000000000
1994-09-12T12:00:00.000000000
1994-09-13T12:00:00.000000000
1994-09-14T12:00:00.000000000
1994-09-15T12:00:00.000000000
1994-09-16T12:00:00.000000000
1994-09-17T12:00:00.000000000
1994-09-18T12:00:00.000000000
1994-09-19T12:00:00.000000000
1994-09-20T12:00:00.000000000
1994-09-21T12:00:00.000000000
1994-09-22T12:00:00.000000000
1994-09-23T12:00:00.000000000
1994-09-24T12:00:00.000000000
1994-09-25T12:00:00.000000000
1994-09-26T12:00:00.000000000
1994-09-27

1996-07-24T12:00:00.000000000
1996-07-25T12:00:00.000000000
1996-07-26T12:00:00.000000000
1996-07-27T12:00:00.000000000
1996-07-28T12:00:00.000000000
1996-07-29T12:00:00.000000000
1996-07-30T12:00:00.000000000
1996-07-31T12:00:00.000000000
1996-08-01T12:00:00.000000000
1996-08-02T12:00:00.000000000
1996-08-03T12:00:00.000000000
1996-08-04T12:00:00.000000000
1996-08-05T12:00:00.000000000
1996-08-06T12:00:00.000000000
1996-08-07T12:00:00.000000000
1996-08-08T12:00:00.000000000
1996-08-09T12:00:00.000000000
1996-08-10T12:00:00.000000000
1996-08-11T12:00:00.000000000
1996-08-12T12:00:00.000000000
1996-08-13T12:00:00.000000000
1996-08-14T12:00:00.000000000
1996-08-15T12:00:00.000000000
1996-08-16T12:00:00.000000000
1996-08-17T12:00:00.000000000
1996-08-18T12:00:00.000000000
1996-08-19T12:00:00.000000000
1996-08-20T12:00:00.000000000
1996-08-21T12:00:00.000000000
1996-08-22T12:00:00.000000000
1996-08-23T12:00:00.000000000
1996-08-24T12:00:00.000000000
1996-08-25T12:00:00.000000000
1996-08-26

1998-06-22T12:00:00.000000000
1998-06-23T12:00:00.000000000
1998-06-24T12:00:00.000000000
1998-06-25T12:00:00.000000000
1998-06-26T12:00:00.000000000
1998-06-27T12:00:00.000000000
1998-06-28T12:00:00.000000000
1998-06-29T12:00:00.000000000
1998-06-30T12:00:00.000000000
1998-07-01T12:00:00.000000000
1998-07-02T12:00:00.000000000
1998-07-03T12:00:00.000000000
1998-07-04T12:00:00.000000000
1998-07-05T12:00:00.000000000
1998-07-06T12:00:00.000000000
1998-07-07T12:00:00.000000000
1998-07-08T12:00:00.000000000
1998-07-09T12:00:00.000000000
1998-07-10T12:00:00.000000000
1998-07-11T12:00:00.000000000
1998-07-12T12:00:00.000000000
1998-07-13T12:00:00.000000000
1998-07-14T12:00:00.000000000
1998-07-15T12:00:00.000000000
1998-07-16T12:00:00.000000000
1998-07-17T12:00:00.000000000
1998-07-18T12:00:00.000000000
1998-07-19T12:00:00.000000000
1998-07-20T12:00:00.000000000
1998-07-21T12:00:00.000000000
1998-07-22T12:00:00.000000000
1998-07-23T12:00:00.000000000
1998-07-24T12:00:00.000000000
1998-07-25

2000-05-21T12:00:00.000000000
2000-05-22T12:00:00.000000000
2000-05-23T12:00:00.000000000
2000-05-24T12:00:00.000000000
2000-05-25T12:00:00.000000000
2000-05-26T12:00:00.000000000
2000-05-27T12:00:00.000000000
2000-05-28T12:00:00.000000000
2000-05-29T12:00:00.000000000
2000-05-30T12:00:00.000000000
2000-05-31T12:00:00.000000000
2000-06-01T12:00:00.000000000
2000-06-02T12:00:00.000000000
2000-06-03T12:00:00.000000000
2000-06-04T12:00:00.000000000
2000-06-05T12:00:00.000000000
2000-06-06T12:00:00.000000000
2000-06-07T12:00:00.000000000
2000-06-08T12:00:00.000000000
2000-06-09T12:00:00.000000000
2000-06-10T12:00:00.000000000
2000-06-11T12:00:00.000000000
2000-06-12T12:00:00.000000000
2000-06-13T12:00:00.000000000
2000-06-14T12:00:00.000000000
2000-06-15T12:00:00.000000000
2000-06-16T12:00:00.000000000
2000-06-17T12:00:00.000000000
2000-06-18T12:00:00.000000000
2000-06-19T12:00:00.000000000
2000-06-20T12:00:00.000000000
2000-06-21T12:00:00.000000000
2000-06-22T12:00:00.000000000
2000-06-23

In [61]:
interpolated_t = interpolated_t[valid_time_indices, :, :, :]

In [35]:
# # creation of q_fit profiles -> for every gridpoint and everyday, calculate the 0-1km ("mixed layer") average q

# # the collection of all these average qs across time for each gridpoint is the xinput to the regression

# # then, for each height, collect all q values for all times -> this is your y_target

# # the lr model is trained on x input and y input and returns a prediction for q values at that height

# # note that q_fit is equivalent to original q profiles below 2km

# interpolated_altitudes = np.arange(0, 4000 + 20, 20)  

# n_times = interpolated_q.shape[0]  

# n_altitudes = len(interpolated_altitudes) 

# n_lats = lats.shape[0]  

# n_lons = lons.shape[0]  

# interpolated_qfit = np.full((n_times, n_altitudes, n_lats, n_lons), np.nan, dtype=np.float32)

# for latidx in range(n_lats):
    
#     for lonidx in range(n_lons):
        
#         print(lats[latidx], lons[lonidx])
        
#         # extracts all q profiles for a given gridpoint across time series 
        
#         q_array = interpolated_q[:, :, latidx, lonidx]  # shape (time, levels)
        
#         # define indices for bottom 1km and 2km
        
#         bottom_1km_idx = interpolated_altitudes <= 1000
        
#         bottom_2km_idx = interpolated_altitudes <= 2000
        
#         target_idx = (interpolated_altitudes > 2000) & (interpolated_altitudes <= 4000)  # heights to predict

#         # compute near-surface q (mean of bottom 1km for every profile in time series)
        
#         near_surface_q = np.mean(q_array[:, bottom_1km_idx], axis=1).reshape(-1, 1) # shape = (time)

#         lr = LinearRegression()

#         # copy q_array and retain observed values up to 2km
        
#         q_fit = q_array.copy()
        
#         q_fit[:, ~bottom_2km_idx] = np.nan  # Keep observed q below 2km

#         # loop over target heights (above 2km) and perform regression
        
#         for j, idx in enumerate(np.where(target_idx)[0]):
            
#             y_target = q_array[:, idx].reshape(-1, 1)  # Shape (2754, 1)
            
#             # train regression model
            
#             lr.fit(near_surface_q, y_target)
            
#             # Predict q_fit values for this height and assign them to the large array
            
#             q_fit[:, idx] = lr.predict(near_surface_q).flatten()

#         # append the q_fit for this lat, lon position into the big array
        
#         interpolated_qfit[:, :, latidx, lonidx] = q_fit

# print(interpolated_qfit.shape)

39.0 -105.0
39.0 -104.75
39.0 -104.5
39.0 -104.25
39.0 -104.0
39.0 -103.75
39.0 -103.5
39.0 -103.25
39.0 -103.0
39.0 -102.75
39.0 -102.5
39.0 -102.25
39.0 -102.0
39.0 -101.75
39.0 -101.5
39.0 -101.25
39.0 -101.0
39.0 -100.75
39.0 -100.5
39.0 -100.25
39.0 -100.0
39.0 -99.75
39.0 -99.5
39.0 -99.25
39.0 -99.0
39.0 -98.75
39.0 -98.5
39.0 -98.25
39.0 -98.0
39.0 -97.75
39.0 -97.5
39.0 -97.25
39.0 -97.0
39.0 -96.75
39.0 -96.5
39.0 -96.25
39.0 -96.0
39.0 -95.75
39.0 -95.5
39.0 -95.25
39.0 -95.0
38.75 -105.0
38.75 -104.75
38.75 -104.5
38.75 -104.25
38.75 -104.0
38.75 -103.75
38.75 -103.5
38.75 -103.25
38.75 -103.0
38.75 -102.75
38.75 -102.5
38.75 -102.25
38.75 -102.0
38.75 -101.75
38.75 -101.5
38.75 -101.25
38.75 -101.0
38.75 -100.75
38.75 -100.5
38.75 -100.25
38.75 -100.0
38.75 -99.75
38.75 -99.5
38.75 -99.25
38.75 -99.0
38.75 -98.75
38.75 -98.5
38.75 -98.25
38.75 -98.0
38.75 -97.75
38.75 -97.5
38.75 -97.25
38.75 -97.0
38.75 -96.75
38.75 -96.5
38.75 -96.25
38.75 -96.0
38.75 -95.75
38.75 -95.5


35.0 -104.75
35.0 -104.5
35.0 -104.25
35.0 -104.0
35.0 -103.75
35.0 -103.5
35.0 -103.25
35.0 -103.0
35.0 -102.75
35.0 -102.5
35.0 -102.25
35.0 -102.0
35.0 -101.75
35.0 -101.5
35.0 -101.25
35.0 -101.0
35.0 -100.75
35.0 -100.5
35.0 -100.25
35.0 -100.0
35.0 -99.75
35.0 -99.5
35.0 -99.25
35.0 -99.0
35.0 -98.75
35.0 -98.5
35.0 -98.25
35.0 -98.0
35.0 -97.75
35.0 -97.5
35.0 -97.25
35.0 -97.0
35.0 -96.75
35.0 -96.5
35.0 -96.25
35.0 -96.0
35.0 -95.75
35.0 -95.5
35.0 -95.25
35.0 -95.0
34.75 -105.0
34.75 -104.75
34.75 -104.5
34.75 -104.25
34.75 -104.0
34.75 -103.75
34.75 -103.5
34.75 -103.25
34.75 -103.0
34.75 -102.75
34.75 -102.5
34.75 -102.25
34.75 -102.0
34.75 -101.75
34.75 -101.5
34.75 -101.25
34.75 -101.0
34.75 -100.75
34.75 -100.5
34.75 -100.25
34.75 -100.0
34.75 -99.75
34.75 -99.5
34.75 -99.25
34.75 -99.0
34.75 -98.75
34.75 -98.5
34.75 -98.25
34.75 -98.0
34.75 -97.75
34.75 -97.5
34.75 -97.25
34.75 -97.0
34.75 -96.75
34.75 -96.5
34.75 -96.25
34.75 -96.0
34.75 -95.75
34.75 -95.5
34.75 -95.25

31.0 -104.5
31.0 -104.25
31.0 -104.0
31.0 -103.75
31.0 -103.5
31.0 -103.25
31.0 -103.0
31.0 -102.75
31.0 -102.5
31.0 -102.25
31.0 -102.0
31.0 -101.75
31.0 -101.5
31.0 -101.25
31.0 -101.0
31.0 -100.75
31.0 -100.5
31.0 -100.25
31.0 -100.0
31.0 -99.75
31.0 -99.5
31.0 -99.25
31.0 -99.0
31.0 -98.75
31.0 -98.5
31.0 -98.25
31.0 -98.0
31.0 -97.75
31.0 -97.5
31.0 -97.25
31.0 -97.0
31.0 -96.75
31.0 -96.5
31.0 -96.25
31.0 -96.0
31.0 -95.75
31.0 -95.5
31.0 -95.25
31.0 -95.0
30.75 -105.0
30.75 -104.75
30.75 -104.5
30.75 -104.25
30.75 -104.0
30.75 -103.75
30.75 -103.5
30.75 -103.25
30.75 -103.0
30.75 -102.75
30.75 -102.5
30.75 -102.25
30.75 -102.0
30.75 -101.75
30.75 -101.5
30.75 -101.25
30.75 -101.0
30.75 -100.75
30.75 -100.5
30.75 -100.25
30.75 -100.0
30.75 -99.75
30.75 -99.5
30.75 -99.25
30.75 -99.0
30.75 -98.75
30.75 -98.5
30.75 -98.25
30.75 -98.0
30.75 -97.75
30.75 -97.5
30.75 -97.25
30.75 -97.0
30.75 -96.75
30.75 -96.5
30.75 -96.25
30.75 -96.0
30.75 -95.75
30.75 -95.5
30.75 -95.25
30.75 -95.0


In [64]:
# Step 1: Define the backup directory in your home folder

backup_dir = os.path.expanduser("~/backup_data")  # Expands ~ to full home directory path

os.makedirs(backup_dir, exist_ok=True)  # create the directory if it doesn't exist

# Step 3: Save multiple arrays in a compressed file

# file_path = os.path.join(backup_dir, "backup_arrays_for_buoyancy.npz")  # Full path to save file

file_path = os.path.join(backup_dir, "backup_arrays_for_buoyancy_12lst_1979_2000.npz")  # Full path to save file

np.savez_compressed(file_path, array1=interpolated_pressure, array2=interpolated_q, array3=interpolated_t)

print(f"Arrays saved to {file_path}")

# Step 4: Verify that the file exists

print("Saved files:", os.listdir(backup_dir))

Arrays saved to /home/annierosen16/backup_data/backup_arrays_for_buoyancy_12lst_1979_2000.npz
Saved files: ['backup_arrays_for_buoyancy.npz', 'backup_metadata_for_buoyancy.npz', 'ctp_hilow_input_arrays.npz', 'backup_arrays_for_advection.npz', 'ml_q_9am.nc', 'backup_arrays_for_buoyancy_15lst.npz', 'backup_metadata_for_buoyancy_15lst.npz', 'backup_arrays_for_buoyancy_15lst_2001_2002_6km_agl.npz', 'backup_metadata_for_buoyancy_15lst_2001_2002_6km_agl.npz', 'backup_2001_2002_lfc_arrays.npz', 'backup_arrays_for_buoyancy_12lst_2001_2025.npz', 'backup_metadata_for_buoyancy_12lst_2001_2025.npz', 'dewpoint_for_ctp_hilow_2001_2025.dat', 'backup_arrays_for_buoyancy_12lst_1979_2000.npz']


In [66]:
times_to_save = times[valid_time_indices]

In [67]:
backup_dir = os.path.expanduser("~/backup_data")

os.makedirs(backup_dir, exist_ok=True)  # Ensure the directory exists

# file_path = os.path.join(backup_dir, "backup_metadata_for_buoyancy.npz")

file_path = os.path.join(backup_dir, "backup_metadata_for_buoyancy_12lst_1979_2000.npz")

np.savez(file_path, 
         
         times=times, 
         
         lats=lats, 
         
         lons=lons)

print(f"Backup saved at {file_path}")

Backup saved at /home/annierosen16/backup_data/backup_metadata_for_buoyancy_12lst_1979_2000.npz
